# Baseline Model Results

Evaluates the logistic regression baseline on aggregate possession features.

**Features used:**
- n_events, n_pass, n_carry, n_dribble, n_pressure, n_attacking_third
- start_x, end_x, progression, total_duration, match_minute_start
- start_zone, end_zone, play_pattern (categorical OHE)

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import (
    roc_curve, precision_recall_curve, roc_auc_score,
    average_precision_score, confusion_matrix, ConfusionMatrixDisplay
)

ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(ROOT))

MODELS = ROOT / 'models' / 'trained'
PROCESSED = ROOT / 'data' / 'processed'

NUMERIC_FEATURES = [
    'n_events','n_pass','n_carry','n_dribble','n_pressure',
    'n_attacking_third','start_x','end_x','progression',
    'total_duration','match_minute_start',
]
CATEGORICAL_FEATURES = ['start_zone','end_zone','play_pattern']

model = joblib.load(MODELS / 'baseline_logreg.pkl')
print('Baseline model loaded')

In [ ]:
def load_split(split):
    match_ids = pd.read_csv(PROCESSED / 'splits' / f'{split}_matches.csv')['match_id'].tolist()
    df = pd.read_parquet(PROCESSED / 'possessions' / 'possessions.parquet')
    return df[df['match_id'].isin(match_ids)].reset_index(drop=True)

val_df = load_split('validation')
test_df = load_split('test')

def get_probs(df):
    X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    return model.predict_proba(X)[:, 1]

val_probs = get_probs(val_df)
test_probs = get_probs(test_df)
print(f'Val  ROC-AUC: {roc_auc_score(val_df.ends_in_shot, val_probs):.4f}')
print(f'Test ROC-AUC: {roc_auc_score(test_df.ends_in_shot, test_probs):.4f}')

## ROC and PR Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for split_name, df_s, probs in [('Validation', val_df, val_probs), ('Test', test_df, test_probs)]:
    y = df_s['ends_in_shot'].values
    fpr, tpr, _ = roc_curve(y, probs)
    p, r, _ = precision_recall_curve(y, probs)
    roc = roc_auc_score(y, probs)
    pr = average_precision_score(y, probs)

    axes[0].plot(fpr, tpr, label=f'{split_name} AUC={roc:.3f}')
    axes[1].plot(r, p, label=f'{split_name} PR-AUC={pr:.3f}')

axes[0].plot([0,1],[0,1],'k--', alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC Curve — Baseline')
axes[0].legend()

axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('PR Curve — Baseline')
axes[1].legend()

plt.tight_layout()
plt.savefig(MODELS / 'baseline_roc_pr.png', dpi=120)
plt.show()

## Per-Competition Breakdown

In [ ]:
print('Validation — per competition:')
for comp, grp in val_df.groupby('competition_label'):
    p = model.predict_proba(grp[NUMERIC_FEATURES + CATEGORICAL_FEATURES])[:, 1]
    y = grp['ends_in_shot'].values
    print(f'  {comp:20s} | n={len(y):4d} | ROC={roc_auc_score(y,p):.4f} | PR={average_precision_score(y,p):.4f}')

## Feature Importance (Logistic Regression Coefficients)

In [ ]:
import numpy as np

clf = model.named_steps['clf']
pre = model.named_steps['preprocessor']

# Get feature names
num_names = NUMERIC_FEATURES
cat_names = pre.transformers_[1][1].get_feature_names_out(CATEGORICAL_FEATURES).tolist()
all_names = num_names + cat_names

coefs = clf.coef_[0]
top_k = 15
idx = np.argsort(np.abs(coefs))[::-1][:top_k]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#ef4444' if c > 0 else '#3b82f6' for c in coefs[idx]]
ax.barh([all_names[i] for i in idx[::-1]], coefs[idx[::-1]], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 15 Logistic Regression Coefficients')
ax.set_xlabel('Coefficient (positive = higher shot prob)')
plt.tight_layout()
plt.savefig(MODELS / 'baseline_feature_importance.png', dpi=120)
plt.show()

## Calibration Curve

In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(6, 5))
y_val = val_df['ends_in_shot'].values
frac_pos, mean_pred = calibration_curve(y_val, val_probs, n_bins=10)
ax.plot(mean_pred, frac_pos, 'o-', label='Baseline', color='#3b82f6')
ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curve — Baseline (Validation)')
ax.legend()
plt.tight_layout()
plt.savefig(MODELS / 'baseline_calibration.png', dpi=120)
plt.show()